# Scribal glyph character spotting - YOLOv8m (stratified split)

Retrains tasks 2, 3 and 4 on the **stratified** split, where all four books
appear in train, validation and test. The legacy split placed the whole
validation set inside book 004, so its figures describe within-book
performance only.

**Nothing measured here is comparable to the numbers in README.md.** Those were
measured on the legacy split. Report this run on its own terms.

Runs on Colab or Kaggle without edits. No credentials are stored in this
notebook: Weights & Biases is off by default and reads a secret only if you
opt in.


## Environment

In [1]:
!pip install -q --upgrade ultralytics pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 7.0 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import os, gc, glob, warnings
import numpy as np
import torch

warnings.filterwarnings("ignore")
os.environ["YOLO_VERBOSE"] = "True"

# Where the data lives.
#   Colab  : mount Drive; datasets sit under MyDrive/
#   Kaggle : attach each dataset; they appear under /kaggle/input/
#   other  : set SCRIBAL_DATA_ROOT yourself

if os.path.exists("/content"):
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/MyDrive"
    RUN_ROOT = "/content/drive/MyDrive/scribal_runs_stratified"
    ENV = "colab"
elif os.path.exists("/kaggle/input"):
    DATA_ROOT = "/kaggle/input"
    RUN_ROOT = "/kaggle/working/runs"
    ENV = "kaggle"
else:
    DATA_ROOT = os.environ["SCRIBAL_DATA_ROOT"]
    RUN_ROOT = os.path.join(DATA_ROOT, "scribal_runs_stratified")
    ENV = "local"

DS2 = os.path.join(DATA_ROOT, "dataset_stratified")
DS3 = os.path.join(DATA_ROOT, "dataset_stratified_task3")
YAML2 = os.path.join(DS2, "scribal-glyph-charspotting-stratified.yaml")
YAML3 = os.path.join(DS3, "scribal-glyph-charspotting-stratified-task3.yaml")

print("env:", ENV)
print("  task 2 data:", DS2)
print("  task 3 data:", DS3)
print("  runs:       ", RUN_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
env: colab
  task 2 data: /content/drive/MyDrive/dataset_stratified
  task 3 data: /content/drive/MyDrive/dataset_stratified_task3
  runs:        /content/drive/MyDrive/scribal_runs_stratified


## Optional: Weights & Biases

Off by default. To enable it, store the key as a Colab secret or a Kaggle
secret named `WANDB_API_KEY` and set `USE_WANDB = True`. Never paste a key into
a cell: this notebook is meant to be shareable as-is. If no key is found the
run proceeds without logging.

In [6]:
USE_WANDB = False
wandb = None

if USE_WANDB:
    key = os.environ.get("WANDB_API_KEY")
    if key is None and ENV == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("WANDB_API_KEY")
        except Exception:
            key = None
    if key is None and ENV == "colab":
        try:
            from google.colab import userdata
            key = userdata.get("WANDB_API_KEY")
        except Exception:
            key = None
    if key:
        import builtins
        import wandb as _wandb
        builtins.RANK = int(os.getenv("RANK", -1))
        _wandb.login(key=key)
        wandb = _wandb
        print("W&B enabled")
    else:
        print("No WANDB_API_KEY found; continuing without logging")
else:
    print("W&B disabled")

W&B disabled


## Guardrails

Two failure modes have already cost this project real time:

1. A run name that collides with an existing directory. Ultralytics silently
   writes to `name2`, while the evaluation cells load `best.pt` from `name` and
   report the **previous** run's numbers.
2. A stale `labels/*.cache`, which makes Ultralytics load the wrong label list
   with no error.

`resolve_run` refuses to reuse a name rather than letting the first one happen
quietly.

In [10]:
def clear_caches(*roots):
    """Remove Ultralytics label caches so a changed split cannot be masked."""
    for root in roots:
        for cache in glob.glob(os.path.join(root, "**", "*.cache"), recursive=True):
            try:
                os.remove(cache)
                print("removed stale cache:", cache)
            except OSError:
                print("could not remove:", cache)


def resolve_run(name):
    """Fail loudly instead of letting Ultralytics auto-suffix the directory."""
    path = os.path.join(RUN_ROOT, name)
    if os.path.exists(path):
        raise SystemExit(
            f"{path} already exists. Ultralytics would write to '{name}2' while "
            f"the evaluation cells load best.pt from '{name}', reporting the OLD "
            "run. Delete that directory or choose a new name."
        )
    return path


def report(tag, metrics):
    print("\n=== " + tag + " ===")
    print("  mAP50    ", round(float(metrics.box.map50), 4))
    print("  mAP50-95 ", round(float(metrics.box.map), 4))
    print("  precision", round(float(np.mean(metrics.box.p)), 4))
    print("  recall   ", round(float(np.mean(metrics.box.r)), 4))
    print("  classes  ", len(metrics.box.p))


if ENV == "kaggle":
    print("Kaggle inputs are read-only; caches are written under /kaggle/working")
else:
    clear_caches(DS2, DS3)

gc.collect()
torch.cuda.empty_cache()

In [12]:
import yaml

for tag, path in (("task 2", YAML2), ("task 3", YAML3)):
    with open(path) as f:
        spec = yaml.safe_load(f)
    print(tag, "->", spec["path"], "| train:", spec["train"],
          "| declared classes:", len(spec["names"]))

task 2 -> /content/drive/MyDrive/dataset_stratified | train: train.txt | declared classes: 54
task 3 -> /content/drive/MyDrive/dataset_stratified_task3 | train: train.txt | declared classes: 54


# Task 2 - baseline, unmodified tiles

The hyperparameters match the legacy run deliberately: 200 epochs,
`patience=50`, `imgsz=512`, batch 16, `fliplr=0.0`. Glyphs are chiral, so a
mirrored `b` is a `d` and flip augmentation would teach the detector the wrong
class. Changing any of these makes the legacy/stratified difference stop being
about the split alone.

In [13]:
from ultralytics import YOLO

RUN2 = "exp_task2_stratified"
resolve_run(RUN2)

model2 = YOLO("yolov8m.pt")
if wandb:
    wandb.init(project="scribal-stratified", name=RUN2)

model2.train(
    project=RUN_ROOT,
    name=RUN2,
    data=YAML2,
    epochs=200,
    imgsz=512,
    patience=50,
    fliplr=0.0,
    amp=True,
)

print("save_dir:", model2.trainer.save_dir)
print("Confirm that matches", os.path.join(RUN_ROOT, RUN2), "before evaluating.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/dataset_stratified/scribal-glyph-charspotting-stratified.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fl

### Validation

`train()` reloads `best.pt` into the model when it finishes, so this reports the
best checkpoint rather than the final epoch. Reading final-epoch numbers off
`last.pt` is what forced the August 2026 retraction, so check the weights line
in the output.

In [14]:
val2 = model2.val(
    project=RUN_ROOT,
    name=RUN2 + "/val",
    data=YAML2,
    imgsz=512,
    batch=16,
    plots=True,
)
report("Task 2 validation (best.pt)", val2)

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 92 layers, 25,871,026 parameters, 0 gradients, 78.9 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.0 ms, read: 68.3±13.6 MB/s, size: 75.9 KB)
val: Scanning /content/drive/MyDrive/dataset_stratified/labels/val.cache... 127 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 127/127 28.0Mit/s 0.0s
val: /content/drive/MyDrive/dataset_stratified/images/val/image_17_15.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified/images/val/image_27_13.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified/images/val/image_27_9.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified/images/val/image_33_11.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified/images/val/image_7_24.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified/images/val/image_8_60.jpg: 1 duplicate labe

### Held-out test

In [15]:
BEST2 = os.path.join(RUN_ROOT, RUN2, "weights", "best.pt")
assert os.path.exists(BEST2), BEST2

test2 = YOLO(BEST2).val(
    project=os.path.join(RUN_ROOT, RUN2),
    name="test",
    data=YAML2,
    split="test",
    imgsz=512,
    batch=16,
    plots=True,
    save=True,
)
report("Task 2 test (best.pt)", test2)

print("\nRecord the instance count Ultralytics printed above, not the label-row")
print("count: exact duplicate rows are dropped at load time.")

if wandb:
    wandb.finish()

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 92 layers, 25,871,026 parameters, 0 gradients, 78.9 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 1.4±0.5 ms, read: 0.2±0.1 MB/s, size: 73.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/dataset_stratified/labels/test... 135 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 135/135 1.2it/s 1:53
val: /content/drive/MyDrive/dataset_stratified/images/test/image_10_21.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified/images/test/image_10_77.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified/images/test/image_19_10.jpg: 2 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified/images/test/image_19_7.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_strat

### Tile predictions, for de-tiling back to page coordinates

In [16]:
from pathlib import Path

imgs2 = sorted(Path(DS2, "images", "test").glob("*.jpg"))
print("test tiles:", len(imgs2))

YOLO(BEST2).predict(
    source=[str(p) for p in imgs2],
    project=os.path.join(RUN_ROOT, RUN2),
    name="predict_task2_stratified",
    conf=0.5,
    iou=0.45,
    imgsz=512,
    batch=16,
    agnostic_nms=True,
    save=True,
    save_conf=True,
    save_txt=True,
)

test tiles: 135

0: 512x512 1 m, 1 q, 1 u, 1 n, 1 i, 1 l, 1 d, 12.7ms
1: 512x512 1 e, 2 ts, 1 a, 4 us, 3 ns, 2 ss, 6 ls, 2 vs, 1 r, 12.7ms
2: 512x512 1 e, 1 t, 3 as, 3 us, 2 ns, 2 ls, 2 os, 12.7ms
3: 512x512 1 q, 3 es, 1 t, 1 a, 1 u, 3 ns, 2 is, 1 l, 2 vs, 1 d, 1 o, 12.7ms
4: 512x512 1 m, 2 es, 1 a, 1 p, 1 n, 1 s, 1 c, 1 o, 1 r, 12.7ms
5: 512x512 2 qs, 1 e, 1 t, 1 a, 1 u, 1 n, 2 is, 1 d, 1 c, 12.7ms
6: 512x512 1 q, 2 es, 2 ts, 3 as, 3 us, 4 ns, 1 s, 2 is, 2 ls, 3 os, 12.7ms
7: 512x512 2 qs, 3 es, 2 ts, 3 as, 1 u, 2 ns, 2 ls, 1 d, 1 r, 1 Q, 12.7ms
8: 512x512 1 q, 3 es, 1 t, 2 as, 4 us, 1 p, 4 ns, 3 is, 3 ls, 1 d, 2 os, 1 r, 1 z, 1 N, 12.7ms
9: 512x512 1 q, 1 t, 4 as, 2 us, 1 n, 1 s, 2 is, 1 l, 1 d, 2 cs, 1 o, 1 r, 12.7ms
10: 512x512 2 qs, 3 es, 1 t, 1 u, 1 n, 1 l, 2 rs, 12.7ms
11: 512x512 2 ms, 2 qs, 3 es, 2 ts, 1 a, 1 u, 1 p, 1 n, 1 b, 1 i, 1 l, 1 o, 1 r, 12.7ms
12: 512x512 1 q, 2 es, 3 ts, 3 as, 3 us, 1 p, 2 ns, 1 S, 12.7ms
13: 512x512 3 es, 1 t, 3 as, 1 u, 1 p, 3 ns, 4 ss, 1 b, 1 i, 

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: None
 names: {0: 'm', 1: 'q', 2: 'e', 3: 'C', 4: 't', 5: 'a', 6: 'u', 7: 'p', 8: 'n', 9: 's', 10: 'f', 11: 'b', 12: 'h', 13: 'i', 14: 'l', 15: 'D', 16: 'v', 17: 'H', 18: 'd', 19: 'c', 20: 'P', 21: 'g', 22: 'o', 23: 'r', 24: 'x', 25: 'j', 26: 'zz_ligature with dachförmiges a', 27: 'z', 28: 'S', 29: 'N', 30: 'T', 31: 'y', 32: 'I', 33: 'E', 34: 'O', 35: 'A', 36: 'unnamed_class_36', 37: 'J', 38: 'Q', 39: 'R', 40: 'F', 41: 'U', 42: 'L', 43: 'B', 44: 'M', 45: 'V', 46: 'w', 47: 'G', 48: 'Y', 49: 'k', 50: 'X', 51: 'K', 52: 'zz', 53: '$'}
 obb: None
 orig_img: array([[[178, 205, 225],
         [181, 208, 228],
         [183, 210, 230],
         ...,
         [206, 221, 230],
         [204, 219, 228],
         [203, 218, 227]],
 
        [[177, 204, 224],
         [179, 206, 226],
         [181, 208, 228],
         ...,
         [202, 217, 226],
    

# Task 3 - page context stripped from the training tiles

Everything outside the labelled boxes is blanked, **in the training tiles
only**. Validation and test remain ordinary page images, so the gap this
measures combines the loss of context with a train/eval distribution shift. It
is not a clean ablation and must not be reported as one.

In [17]:
RUN3 = "exp_task3_stratified"
resolve_run(RUN3)

gc.collect()
torch.cuda.empty_cache()

model3 = YOLO("yolov8m.pt")
if wandb:
    wandb.init(project="scribal-stratified", name=RUN3)

model3.train(
    project=RUN_ROOT,
    name=RUN3,
    data=YAML3,
    epochs=200,
    imgsz=512,
    patience=50,
    fliplr=0.0,
    amp=True,
)

print("save_dir:", model3.trainer.save_dir)

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/dataset_stratified_task3/scribal-glyph-charspotting-stratified-task3.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, m

### Validation

In [18]:
val3 = model3.val(
    project=RUN_ROOT,
    name=RUN3 + "/val",
    data=YAML3,
    imgsz=512,
    batch=16,
    plots=True,
)
report("Task 3 validation (best.pt)", val3)

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 92 layers, 25,871,026 parameters, 0 gradients, 78.9 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 68.2±22.1 MB/s, size: 79.0 KB)
val: Scanning /content/drive/MyDrive/dataset_stratified_task3/labels/val.cache... 127 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 127/127 33.3Mit/s 0.0s
val: /content/drive/MyDrive/dataset_stratified_task3/images/val/image_17_15.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified_task3/images/val/image_27_13.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified_task3/images/val/image_27_9.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified_task3/images/val/image_33_11.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified_task3/images/val/image_7_24.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified_task3/i

### Held-out test

In [19]:
BEST3 = os.path.join(RUN_ROOT, RUN3, "weights", "best.pt")
assert os.path.exists(BEST3), BEST3

test3 = YOLO(BEST3).val(
    project=os.path.join(RUN_ROOT, RUN3),
    name="test",
    data=YAML3,
    split="test",
    imgsz=512,
    batch=8,
    plots=True,
    save=True,
)
report("Task 3 test (best.pt)", test3)

if wandb:
    wandb.finish()

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 92 layers, 25,871,026 parameters, 0 gradients, 78.9 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 1.3±0.4 ms, read: 0.2±0.1 MB/s, size: 73.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/dataset_stratified_task3/labels/test... 135 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 135/135 1.3it/s 1:47
val: /content/drive/MyDrive/dataset_stratified_task3/images/test/image_10_21.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified_task3/images/test/image_10_77.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified_task3/images/test/image_19_10.jpg: 2 duplicate labels removed
val: /content/drive/MyDrive/dataset_stratified_task3/images/test/image_19_7.jpg: 1 duplicate labels removed
val: /conte

### Tile predictions

In [20]:
imgs3 = sorted(Path(DS3, "images", "test").glob("*.jpg"))
print("test tiles:", len(imgs3))

YOLO(BEST3).predict(
    source=[str(p) for p in imgs3],
    project=os.path.join(RUN_ROOT, RUN3),
    name="predict_task3_stratified",
    conf=0.5,
    iou=0.45,
    imgsz=512,
    batch=8,
    agnostic_nms=True,
    save=True,
    save_conf=True,
    save_txt=True,
)

test tiles: 135

0: 512x512 1 m, 1 q, 3 es, 1 t, 2 us, 1 p, 2 ns, 2 ss, 1 h, 2 is, 1 l, 3 ds, 12.9ms
1: 512x512 5 es, 6 ts, 2 as, 7 us, 1 p, 3 ns, 3 ss, 2 fs, 1 i, 6 ls, 2 vs, 2 ds, 1 c, 2 os, 2 rs, 12.9ms
2: 512x512 2 ms, 4 es, 4 ts, 4 as, 5 us, 2 ns, 1 s, 2 fs, 1 h, 5 is, 3 ls, 1 v, 3 ds, 2 cs, 2 os, 2 rs, 1 z, 1 Q, 12.9ms
3: 512x512 1 m, 3 qs, 6 es, 4 ts, 2 as, 2 us, 5 ns, 1 s, 6 is, 2 ls, 2 vs, 3 ds, 3 os, 1 r, 2 zs, 12.9ms
4: 512x512 1 m, 1 q, 3 es, 2 ts, 2 as, 1 u, 2 ps, 1 n, 1 s, 1 f, 1 b, 3 is, 1 d, 1 c, 1 o, 2 rs, 12.9ms
5: 512x512 2 qs, 3 es, 2 ts, 2 as, 3 us, 2 ps, 2 ns, 1 h, 4 is, 1 l, 1 d, 1 c, 1 r, 12.9ms
6: 512x512 1 m, 2 qs, 5 es, 4 ts, 4 as, 7 us, 1 p, 5 ns, 1 s, 1 f, 4 is, 4 ls, 3 ds, 6 os, 1 r, 12.9ms
7: 512x512 2 qs, 5 es, 3 ts, 4 as, 5 us, 4 ns, 3 fs, 4 is, 4 ls, 7 ds, 2 os, 1 r, 1 z, 1 Q, 12.9ms
8: 512x512 3 qs, 4 es, 3 ts, 2 as, 6 us, 1 p, 4 ns, 1 s, 6 is, 5 ls, 6 ds, 2 os, 3 rs, 1 z, 1 N, 12.9ms
9: 512x512 1 m, 3 qs, 1 t, 4 as, 3 us, 1 n, 1 s, 5 is, 1 l, 2 ds, 2

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: None
 names: {0: 'm', 1: 'q', 2: 'e', 3: 'C', 4: 't', 5: 'a', 6: 'u', 7: 'p', 8: 'n', 9: 's', 10: 'f', 11: 'b', 12: 'h', 13: 'i', 14: 'l', 15: 'D', 16: 'v', 17: 'H', 18: 'd', 19: 'c', 20: 'P', 21: 'g', 22: 'o', 23: 'r', 24: 'x', 25: 'j', 26: 'zz_ligature with dachförmiges a', 27: 'z', 28: 'S', 29: 'N', 30: 'T', 31: 'y', 32: 'I', 33: 'E', 34: 'O', 35: 'A', 36: 'unnamed_class_36', 37: 'J', 38: 'Q', 39: 'R', 40: 'F', 41: 'U', 42: 'L', 43: 'B', 44: 'M', 45: 'V', 46: 'w', 47: 'G', 48: 'Y', 49: 'k', 50: 'X', 51: 'K', 52: 'zz', 53: '$'}
 obb: None
 orig_img: array([[[178, 205, 225],
         [181, 208, 228],
         [183, 210, 230],
         ...,
         [206, 221, 230],
         [204, 219, 228],
         [203, 218, 227]],
 
        [[177, 204, 224],
         [179, 206, 226],
         [181, 208, 228],
         ...,
         [202, 217, 226],
    

# Task 4 - label completeness

This inverts the augmentations, as the labelled boxes are blanked and the rest of
the page is kept, then task 3's detector runs over those training tiles. The
question is about the ground truth rather than about context - the annotations
are sparse, so does the detector surface character instances that were never
labelled?

No ground truth exists for those instances, so the stage is **qualitative by
construction. No number from it may be quoted.** It also uses task 3's model,
which is the weaker of the two.

In [21]:
imgs4 = sorted(Path(DS3, "task4", "images").glob("*.jpg"))
print("blanked training tiles:", len(imgs4))

gc.collect()
torch.cuda.empty_cache()

YOLO(BEST3).predict(
    source=[str(p) for p in imgs4],
    project=os.path.join(RUN_ROOT, RUN3),
    name="predict_task4_stratified",
    conf=0.5,
    iou=0.45,
    imgsz=512,
    batch=1,
    half=True,
    agnostic_nms=True,
    save=True,
    save_conf=True,
    save_txt=True,
)

torch.cuda.empty_cache()

blanked training tiles: 387
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

0: 512x512 9 ms, 2 qs, 20 es, 3 ts, 16 as, 10 us, 11 ps, 11 ns, 7 ss, 4 fs, 4 bs, 2 hs, 17 is, 7 ls, 8 ds, 1 c, 6 gs, 8 os, 4 rs, 4 xs, 1 w, 6.5ms
1: 512x512 7 ms, 1 q, 10 es, 2 ts, 9 as, 3 us, 9 ps, 7 ns, 5 ss, 2 fs, 1 b, 2 hs, 11 is, 2 ls, 1 v, 5 ds, 1 c, 2 gs, 9 os, 1 x, 1 A, 1 w, 6.5ms
2: 512x512 8 ms, 12 es, 2 ts, 5 as, 5 us, 4 ps, 6 ns, 4 ss, 3 fs, 1 b, 1 h, 5 is, 5 ds, 1 c, 1 g, 3 os, 1 r, 1 x, 6.5ms
3: 512x512 3 ms, 9 es, 3 ts, 6 as, 2 us, 1 p, 3 ns, 2 ss, 2 fs, 3 hs, 5 is, 1 l, 1 d, 1 c, 1 g, 5 os, 1 A, 6.5ms
4: 512x512 7 ms, 3 qs, 9 es, 2 Cs, 4 ts, 12 as, 5 us, 1 p, 5 ns, 5 ss, 2 fs, 4 bs, 1 h, 9 is, 2 ls, 1 d, 1 c, 5 os, 2 rs, 1 z, 6.5ms
5: 512x512 6 ms, 7 es, 2 Cs, 2 ts, 3 as, 4 us, 2 ps, 4 ns, 5 ss, 8 is, 2 ls, 3 os, 1 x, 6.5ms
6: 512x512 15 ms, 4 qs, 33 es, 2 Cs, 5 ts, 17 as, 7 us, 5 ps, 12 ns, 9 ss, 4 fs, 7 bs, 3 hs, 20 is, 5 ls, 3 ds, 2 cs, 11 os, 1 r,

## Summary to record

Copy this table into the README, together with the best epoch and stop epoch
from each run's `results.csv`.

In [24]:
header = ("run", "mAP50", "mAP50-95", "prec", "recall", "cls")
print("%10s %8s %9s %8s %8s %5s" % header)

for tag, m in (("task2 val", val2), ("task2 test", test2),
               ("task3 val", val3), ("task3 test", test3)):
    print("%10s %8.4f %9.4f %8.4f %8.4f %5d" % (
        tag,
        float(m.box.map50),
        float(m.box.map),
        float(np.mean(m.box.p)),
        float(np.mean(m.box.r)),
        len(m.box.p),
    ))

       run    mAP50  mAP50-95     prec   recall   cls
 task2 val   0.6715    0.6220   0.5682   0.6910    31
task2 test   0.7112    0.6584   0.5867   0.7297    31
 task3 val   0.3857    0.3191   0.3413   0.6350    31
task3 test   0.4176    0.3408   0.3444   0.7227    31
